# 🔍 Ethereum Fraud Detection — Model Training

Train a **GradientBoostingClassifier** on the [Kaggle Ethereum Fraud Detection Dataset](https://github.com/Vagif12/Ethereum-Fraud-Detection) (9,841 real accounts, 50+ features).

The trained model integrates with the Crypto Wallet Analyzer risk engine.

**Outputs:**
- Classification report (precision / recall / F1)
- Feature importance ranking
- Downloadable `ml_model.joblib`

## 1. Setup

In [ ]:
# These are pre-installed in Colab, but just in case:
!pip install -q scikit-learn pandas numpy joblib matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

print("Libraries loaded ✓")

## 2. Load Dataset

Upload `ethereum_fraud_dataset.csv` or mount Google Drive.

In [ ]:
# Option A: Upload directly
from google.colab import files
uploaded = files.upload()  # Upload ethereum_fraud_dataset.csv
DATASET_PATH = list(uploaded.keys())[0]
print(f"Using uploaded file: {DATASET_PATH}")

In [ ]:
# Option B (alternative): Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/ethereum_fraud_dataset.csv'

In [ ]:
df = pd.read_csv(DATASET_PATH)
df.columns = df.columns.str.strip()
print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['FLAG'].value_counts())
print(f"\nFraud ratio: {df['FLAG'].mean():.2%}")
df.head()

## 3. Feature Engineering

Map Kaggle columns → our 17 `WalletAggregates` features.

In [ ]:
ML_FEATURE_NAMES = [
    "total_tx_count",
    "total_volume",
    "unique_interacted_addresses",
    "avg_tx_value",
    "active_days",
    "max_tx_value",
    "min_tx_value",
    "std_tx_value",
    "incoming_tx_count",
    "outgoing_tx_count",
    "total_incoming_volume",
    "total_outgoing_volume",
    "net_flow",
    "avg_time_between_tx",
    "max_single_address_volume",
    "error_tx_count",
    "error_tx_ratio",
]

mapped = pd.DataFrame()

mapped["total_tx_count"] = df["total transactions (including tnx to create contract"]
mapped["total_volume"] = df["total Ether sent"] + df["total ether received"]
mapped["unique_interacted_addresses"] = df["Unique Sent To Addresses"] + df["Unique Received From Addresses"]
mapped["avg_tx_value"] = np.where(
    mapped["total_tx_count"] > 0,
    mapped["total_volume"] / mapped["total_tx_count"],
    0.0,
)
mapped["active_days"] = np.clip(
    (df["Time Diff between first and last (Mins)"] / 1440.0).round().astype(int),
    a_min=0, a_max=None,
)
mapped["max_tx_value"] = np.maximum(df["max val sent"], df["max value received"])

min_sent = df["min val sent"]
min_recv = df["min value received"]
mapped["min_tx_value"] = np.where(
    (min_sent > 0) & (min_recv > 0),
    np.minimum(min_sent, min_recv),
    np.maximum(min_sent, min_recv),
)

avg_sent = df["avg val sent"]
avg_recv = df["avg val received"]
max_sent = df["max val sent"]
max_recv = df["max value received"]
combined_avg = (avg_sent + avg_recv) / 2
combined_max = np.maximum(max_sent, max_recv)
mapped["std_tx_value"] = np.abs(combined_max - combined_avg)

mapped["incoming_tx_count"] = df["Received Tnx"]
mapped["outgoing_tx_count"] = df["Sent tnx"]
mapped["total_incoming_volume"] = df["total ether received"]
mapped["total_outgoing_volume"] = df["total Ether sent"]
mapped["net_flow"] = df["total ether received"] - df["total Ether sent"]

avg_min_sent = df["Avg min between sent tnx"].fillna(0)
avg_min_recv = df["Avg min between received tnx"].fillna(0)
mapped["avg_time_between_tx"] = ((avg_min_sent + avg_min_recv) / 2) * 60

mapped["max_single_address_volume"] = df["max val sent"]
mapped["error_tx_count"] = 0
mapped["error_tx_ratio"] = 0.0

mapped = mapped.replace([np.inf, -np.inf], 0.0).fillna(0.0)

X = mapped[ML_FEATURE_NAMES]
y = df["FLAG"]

print(f"Feature matrix shape: {X.shape}")
X.describe()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Compute sample weights for class imbalance
class_counts = y_train.value_counts()
total = len(y_train)
weight_map = {cls: total / (len(class_counts) * count) for cls, count in class_counts.items()}
sample_weights = y_train.map(weight_map)

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")
print(f"Class weight map: {weight_map}")

## 5. Train the Model

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    subsample=0.8,
    min_samples_leaf=10,
)

model.fit(X_train, y_train, sample_weight=sample_weights)
print("Model trained ✓")

## 6. Evaluate

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=["Legit (0)", "Fraud (1)"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'GBM (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature Importance Ranking:")
print("-" * 50)
for rank, idx in enumerate(indices, 1):
    print(f"  {rank:2d}. {ML_FEATURE_NAMES[idx]:<35s} {importances[idx]:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 8))
sorted_names = [ML_FEATURE_NAMES[i] for i in indices]
sorted_imps = importances[indices]

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(sorted_names)))
ax.barh(range(len(sorted_names)), sorted_imps[::-1], color=colors)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names[::-1])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance — Gradient Boosted Decision Tree')
plt.tight_layout()
plt.show()

## 8. Save & Download Model

In [ ]:
MODEL_PATH = 'ml_model.joblib'
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH} ✓")

import os
print(f"Model size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

In [ ]:
# Download to your machine
from google.colab import files
files.download(MODEL_PATH)
print("Download started — place the file at:")
print("  python-server/app/processors/ml_model.joblib")

## 9. Quick Inference Test

Test the model on a sample to see how it predicts + explain.

In [ ]:
# Pick a random fraud account from test set
fraud_indices = y_test[y_test == 1].index
if len(fraud_indices) > 0:
    sample_idx = fraud_indices[0]
    sample = X_test.loc[[sample_idx]]
    prob = model.predict_proba(sample)[0]

    print(f"Sample account (actual: FRAUD)")
    print(f"  P(legit):  {prob[0]:.4f}")
    print(f"  P(fraud):  {prob[1]:.4f}")
    print(f"  Risk score: {prob[1] * 100:.1f}/100")

    # Feature contributions for this prediction
    fv = sample.values[0]
    fv_abs = np.abs(fv)
    fv_max = fv_abs.max()
    fv_norm = fv_abs / fv_max if fv_max > 0 else fv_abs
    raw = importances * fv_norm
    total = raw.sum()

    print(f"\n  Top contributing features:")
    contrib = {name: round(float(val / total * 100), 2) for name, val in zip(ML_FEATURE_NAMES, raw)}
    for name, pct in sorted(contrib.items(), key=lambda x: -x[1])[:5]:
        print(f"    {name:<35s} {pct:.1f}%")